# 01_Dataset_TextRank

P10 — Abstractive Summarisation of Hindi News


In [20]:
# Add your implementation here.
!pip -q install networkx scikit-learn rouge-score

import re
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("TextRank libraries imported successfully.")

TextRank libraries imported successfully.


In [21]:
import pandas as pd

print("Loading XL-Sum Hindi dataset from Hugging Face auto-converted Parquet splits...")
# Valid direct Hugging Face parquet URLs from the system conversion branch
train_url = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/refs%2Fconvert%2Fparquet/hindi/train/0000.parquet"
val_url = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/refs%2Fconvert%2Fparquet/hindi/validation/0000.parquet"
test_url = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/refs%2Fconvert%2Fparquet/hindi/test/0000.parquet"

train_df = pd.read_parquet(train_url)
val_df = pd.read_parquet(val_url)
test_df = pd.read_parquet(test_url)

print("Dataset loaded successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

# Display the first row to confirm
display(train_df.head(1))

Loading XL-Sum Hindi dataset from Hugging Face auto-converted Parquet splits...
Dataset loaded successfully!
Train shape: (63000, 5)
Validation shape: (8847, 5)
Test shape: (8847, 5)


,id,url,title,summary,text
0,international-50010000,https://www.bbc.com/hindi/international-50010000,चीन-नेपाल में गहराती दोस्ती पर इतना शांत क्यों...,चीन के राष्ट्रपति शी जिनपिंग नेपाल के दौरे पर ...,नेपाल में इस दौरे की भरपूर तैयारियां चल रही है...


In [22]:
# Print the column names and sample data from the training set
print("Columns in train_df:", train_df.columns.tolist())
display(train_df[['title', 'summary', 'text']].head(1))

Columns in train_df: ['id', 'url', 'title', 'summary', 'text']


,title,summary,text
0,चीन-नेपाल में गहराती दोस्ती पर इतना शांत क्यों...,चीन के राष्ट्रपति शी जिनपिंग नेपाल के दौरे पर ...,नेपाल में इस दौरे की भरपूर तैयारियां चल रही है...


In [23]:
def split_sentences(text):
    text = str(text).strip()

    sentences = re.split(
        r"(?<=[।!?])\s+|(?<=[.!?])\s+|\n+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

In [24]:
sample = train_df.iloc[0]["text"]

sentences = split_sentences(sample)

print("Number of sentences:", len(sentences))

for i, sentence in enumerate(sentences[:5], start=1):
    print(f"{i}. {sentence}")

Number of sentences: 99
1. नेपाल में इस दौरे की भरपूर तैयारियां चल रही हैं.
2. इसकी ख़ास वजह भी है क्योंकि 23 साल बाद कोई चीनी राष्ट्रपति नेपाल पहुंच रहा है.
3. नेपाल के प्रधानमंत्री केपी शर्मा ओली के विदेश मामलों के सलाहकार डॉ.
4. राजन भट्टाराई ने इस दौरे को ऐतिहासिक बताया है.
5. उन्होंने बताया कि चीन के राष्ट्रपति के साथ उनका एक प्रतिनिधि दल भी होगा.


In [25]:
def textrank_summary(text, max_sentences=3):
    sentences = split_sentences(text)

    if not sentences:
        return ""

    if len(sentences) <= max_sentences:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b"
    )

    try:
        sentence_vectors = vectorizer.fit_transform(sentences)
    except ValueError:
        return " ".join(sentences[:max_sentences])

    similarity_matrix = cosine_similarity(sentence_vectors)

    np.fill_diagonal(similarity_matrix, 0)

    graph = nx.from_numpy_array(similarity_matrix)

    scores = nx.pagerank(graph, weight="weight")

    ranked_sentences = sorted(
        range(len(sentences)),
        key=lambda i: scores[i],
        reverse=True
    )

    selected_indices = sorted(
        ranked_sentences[:max_sentences]
    )

    summary = " ".join(
        sentences[i] for i in selected_indices
    )

    return summary

In [26]:
sample_article = train_df.iloc[0]["text"]
reference_summary = train_df.iloc[0]["summary"]

generated_summary = textrank_summary(
    sample_article,
    max_sentences=3
)

print("ARTICLE:")
print(sample_article[:1500])

print("\nHUMAN REFERENCE SUMMARY:")
print(reference_summary)

print("\nTEXTRANK SUMMARY:")
print(generated_summary)

ARTICLE:
नेपाल में इस दौरे की भरपूर तैयारियां चल रही हैं. इसकी ख़ास वजह भी है क्योंकि 23 साल बाद कोई चीनी राष्ट्रपति नेपाल पहुंच रहा है. नेपाल के प्रधानमंत्री केपी शर्मा ओली के विदेश मामलों के सलाहकार डॉ. राजन भट्टाराई ने इस दौरे को ऐतिहासिक बताया है. उन्होंने बताया कि चीन के राष्ट्रपति के साथ उनका एक प्रतिनिधि दल भी होगा. दोनों देशों के बीच कई समझौते होने की बात भी कही जा रही है. चीन के राष्ट्रपति और नेपाल के प्रधानमंत्री के बीच आधिकारिक बैठक भी तय हुई है. समाप्त नेपाल जाने से पहले चीन के राष्ट्रपति भारत में प्रधानमंत्री नरेंद्र मोदी के साथ अनौपचारिक मुलाक़ात करेंगे. इससे पहले शी जिनपिंग पाकिस्तान के प्रधानमंत्री इमरान ख़ान के साथ भी बैठक कर चुके हैं. चीन का प्रभाव दक्षिण एशिया में लगातार बढ़ रहा है. वो चाहे नेपाल, श्रीलंका, पाकिस्तान या बांग्लादेश हो. हर जगह चीन की मौजूदगी बढ़ी है. ये सभी देश चीन की वन बेल्ट वन रोड परियोजना में शामिल हो गए हैं. दूसरी तरफ़ भारत इस परियोजना के पक्ष में नहीं है. नेपाल की राष्ट्रपति विद्या देवी भंडारी और चीन के राष्ट्रपति शी जिनपिंग नेपाल की तरफ़ बढ़ता च

In [28]:
print("Test set size:", len(test_df))
print("Columns:", test_df.columns.tolist())

display(test_df.head())

Test set size: 8847
Columns: ['id', 'url', 'title', 'summary', 'text']


,id,url,title,summary,text
0,081027_sadhvi_test_pp,https://www.bbc.com/hindi/regionalnews/story/2...,साध्वी का ब्रेन मैपिंग और नॉर्को टेस्ट,मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्व...,नासिक की एक अदालत ने महाराष्ट्र पुलिस के आतंकव...
1,070929_pakistan_protest,https://www.bbc.com/hindi/regionalnews/story/2...,"मुशर्रफ़ का पर्चा मंज़ूर, प्रदर्शनों पर लाठी",पाकिस्तान के चुनाव आयोग में शनिवार को राष्ट्रप...,इस बीच चुनाव आयोग ने शनिवार को राष्ट्रपति परवे...
2,international-45650689,https://www.bbc.com/hindi/international-45650689,इतनी जल्दी फ़्रांस में ही चमक क्यों खोने लगे र...,फ़्रांस के राष्ट्रपति इमैनुएल मैक्रों की पहचान...,16 महीने के कार्यकाल पूरे होने के बावजूद राष्ट...
3,150415_iranian_nuclear_deal_public_humour_rns,https://www.bbc.com/hindi/international/2015/0...,'आइए व्हाइट हाउस में जुमे की नमाज़ पढ़ें',"""ओबामाः अब रिश्ते अच्छे हैं. आइए अब व्हाइट हाउ...","""या ख़ुदा! अब हम अमरीका आ जा सकते हैं! मैं पहन..."
4,040328_israel,https://www.bbc.com/hindi/news/story/2004/03/0...,इसराइल की ख़ुफ़िया एजेंसियों की खिंचाई,इसराइल में एक संसदीय रिपोर्ट में देश की ख़ुफ़ि...,इसराइल की ख़ुफ़िया एजेंसी मोसाद को दुनिया भर म...


In [29]:
test_sample = test_df.head(100).copy()

sample_predictions = []

for i, row in enumerate(test_sample.itertuples(index=False), start=1):
    prediction = textrank_summary(row.text, max_sentences=3)

    sample_predictions.append({
        "article_id": str(row.id),
        "reference": row.summary,
        "prediction": prediction
    })

    if i % 20 == 0:
        print(f"Processed {i}/100")

sample_predictions_df = pd.DataFrame(sample_predictions)

display(sample_predictions_df.head())

Processed 20/100
Processed 40/100
Processed 60/100
Processed 80/100
Processed 100/100


,article_id,reference,prediction
0,081027_sadhvi_test_pp,मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्व...,हालाँकि सरकारी वकील अजय मिश्रा ने यह जानकारी न...
1,070929_pakistan_protest,पाकिस्तान के चुनाव आयोग में शनिवार को राष्ट्रप...,इस बीच चुनाव आयोग ने शनिवार को राष्ट्रपति परवे...
2,international-45650689,फ़्रांस के राष्ट्रपति इमैनुएल मैक्रों की पहचान...,इन आंकड़ों पर साइंसेज़ पो यूनिवर्सिटी के प्रोफ...
3,150415_iranian_nuclear_deal_public_humour_rns,"""ओबामाः अब रिश्ते अच्छे हैं. आइए अब व्हाइट हाउ...","एक चुटकुले में कहा गया है, ""जैसे ओबामा के नाम ..."
4,040328_israel,इसराइल में एक संसदीय रिपोर्ट में देश की ख़ुफ़ि...,इसराइल की एक संसदीय समिति ने जाँच के बाद कहा ह...


In [32]:
predictions_df = sample_predictions_df.copy()

print("Columns:", predictions_df.columns.tolist())
print("Total rows:", len(predictions_df))
print("Duplicate IDs:", predictions_df["article_id"].duplicated().sum())
print("Missing references:", predictions_df["reference"].isna().sum())
print("Missing predictions:", predictions_df["prediction"].isna().sum())

assert predictions_df.columns.tolist() == [
    "article_id", "reference", "prediction"
]

assert predictions_df["article_id"].is_unique
assert predictions_df["prediction"].notna().all()

display(predictions_df.head())

Columns: ['article_id', 'reference', 'prediction']
Total rows: 100
Duplicate IDs: 0
Missing references: 0
Missing predictions: 0


,article_id,reference,prediction
0,081027_sadhvi_test_pp,मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्व...,हालाँकि सरकारी वकील अजय मिश्रा ने यह जानकारी न...
1,070929_pakistan_protest,पाकिस्तान के चुनाव आयोग में शनिवार को राष्ट्रप...,इस बीच चुनाव आयोग ने शनिवार को राष्ट्रपति परवे...
2,international-45650689,फ़्रांस के राष्ट्रपति इमैनुएल मैक्रों की पहचान...,इन आंकड़ों पर साइंसेज़ पो यूनिवर्सिटी के प्रोफ...
3,150415_iranian_nuclear_deal_public_humour_rns,"""ओबामाः अब रिश्ते अच्छे हैं. आइए अब व्हाइट हाउ...","एक चुटकुले में कहा गया है, ""जैसे ओबामा के नाम ..."
4,040328_israel,इसराइल में एक संसदीय रिपोर्ट में देश की ख़ुफ़ि...,इसराइल की एक संसदीय समिति ने जाँच के बाद कहा ह...
